In [13]:
# Instalar bibliotecas
!pip install -q -U transformers accelerate bitsandbytes Pillow==9.5.0 requests
from huggingface_hub import login
import torch

In [14]:
# Mudança de classe
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image, UnidentifiedImageError
import requests
import io
print("Bibliotecas instaladas e importadas com sucesso!")


Bibliotecas instaladas e importadas com sucesso!


In [15]:
# Login Hugging Face
print("\nETAPA 2: Fazendo login no Hugging Face...")
print("""Certifique-se de ter aceitado os termos de uso para
'google/medgemma-4b-it' na página do Hugging Face.""")
login()
print("Login no Hugging Face concluído.")


ETAPA 2: Fazendo login no Hugging Face...
Certifique-se de ter aceitado os termos de uso para
'google/medgemma-4b-it' na página do Hugging Face.
Login no Hugging Face concluído.


In [16]:
#Carregamento do Processador e Modelo com a nova classe de modelo)
print("\nETAPA 3: Carregando o Processador e o Modelo...")
model_id = "google/medgemma-4b-it" # Renomeado para model_id para consistência com o exemplo
print(f"Carregando o processador para o modelo: {model_id}")
try:
    processor = AutoProcessor.from_pretrained(model_id,
    trust_remote_code=True) # trust_remote_code ainda é bom
    print("Processador carregado com sucesso.")
except Exception as e:
    print(f"Erro CRÍTICO ao carregar o processador: {e}")
    raise
print(f"\nCarregando o modelo: {model_id}")
try:
    # --- MUDANÇA IMPORTANTE NA CLASSE DO MODELO ---
    model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto", # device_map="auto" para distribuir na GPU
    trust_remote_code=True # trust_remote_code se o modelo tiver código customizado
    )
    print("Modelo carregado com sucesso na GPU.")
except Exception as e:
    print(f"Erro CRÍTICO ao carregar o modelo: {e}")
    raise


ETAPA 3: Carregando o Processador e o Modelo...
Carregando o processador para o modelo: google/medgemma-4b-it
Processador carregado com sucesso.

Carregando o modelo: google/medgemma-4b-it


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Modelo carregado com sucesso na GPU.


In [17]:
# Definição da função para interagir com MedGemma
print("""\nETAPA 4: Definindo a função 'ask_medgemma_multimodal_new'...""")

def ask_medgemma_multimodal_new(texto_pergunta_usuario, url_imagem,
                                 system_prompt="Você é um radiologista especialista.",
                                 max_new_tokens=500):
    """
    Envia uma pergunta (texto e imagem de URL) para o modelo
    MedGemma usando o método de chat_template e imprime a resposta.
    """
    print(f"""\n Enviando pergunta multimodal (imagem de URL: {url_imagem}) para MedGemma (aguarde)...""")
    try:
        print(f"Baixando imagem de: {url_imagem}")
        # Adicionar User-Agent pode ajudar com alguns servidores de imagem
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"}
        response = requests.get(url_imagem, headers=headers, stream=True)
        response.raise_for_status()
        imagem_obj = Image.open(io.BytesIO(response.content)).convert("RGB") # Garantir que é RGB
        print("Imagem baixada e carregada com sucesso.")
    except requests.exceptions.RequestException as e:
        print(f"Erro ao baixar a imagem do URL '{url_imagem}': {e}")
        return
    except UnidentifiedImageError:
        print(f"Erro: Não foi possível identificar o arquivo de imagem no URL.")
        return
    except Exception as e:
        print(f"Erro ao carregar a imagem do URL: {e}")
        return

    # Constrói a estrutura de mensagens
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompt}]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": texto_pergunta_usuario},
                {"type": "image", "image": imagem_obj} # Passa o objeto PIL Image
            ]
        }
    ]
    try:
        print("Processando input com apply_chat_template...")
        # O dtype=torch.bfloat16 aqui pode não ser necessário se o modelo já está em bfloat16
        # mas não deve prejudicar.
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True, # Importante para instruir o modelo a gerar uma resposta
            tokenize=True, # Queremos os IDs dos tokens
            return_dict=True, # Retorna um dicionário com input_ids, attention_mask, etc.
            return_tensors="pt" # Retorna tensores PyTorch
        ).to(model.device) # Mover todos os tensores para o dispositivo do modelo
        # Se 'pixel_values' estiverem em um dtype diferente (ex: float32) e o modelo espera bfloat16,
        # pode ser necessário converter inputs['pixel_values'] especificamente.
        # No entanto, o processador deve lidar com isso idealmente.
        # Se o modelo está em bfloat16, os inputs também deveriam estar.
        # Forçar dtype para todos os inputs:
        # for key in inputs:
        # if torch.is_tensor(inputs[key]):
        # inputs[key] = inputs[key].to(dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)

        # Verificar se pixel_values existem
        if 'pixel_values' not in inputs or inputs['pixel_values'] is None:
            print("ALERTA: 'pixel_values' não foram encontrados nos inputs processados. A imagem pode não ter sido incluída.")
            # Adicionar um pixel_values dummy se não existir pode evitar erros, mas a resposta será apenas textual
            # inputs['pixel_values'] = torch.zeros((1, 3, 224, 224), dtype=model.dtype).to(model.device) # Exemplo de dummy
        else:
            # Garante que pixel_values também esteja no device e dtype corretos
            inputs['pixel_values'] = \
                inputs['pixel_values'].to(device=model.device, dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)
    except Exception as e:
        print(f"Erro ao processar o input com apply_chat_template: {e}")
        return

    input_len = inputs["input_ids"].shape[-1] # Comprimento dos tokens de entrada

    try:
        print("Gerando resposta do modelo...")
        with torch.inference_mode(): # Importante para economizar memória e acelerar na inferência
            generation_output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False # O exemplo usa False, para decodificação greedy. Podemos mudar para True se quisermos amostragem.
                # Se do_sample=True:
                # temperature=0.7,
                # top_p=0.95,
                # top_k=50,
            )
        # Isolar apenas os tokens gerados (não o prompt de entrada)
        generated_ids = generation_output[0][input_len:]
        decoded_response = processor.decode(generated_ids, skip_special_tokens=True)
        print("\nResposta do MedGemma (Análise da Imagem):")
        print(decoded_response)
    except torch.cuda.OutOfMemoryError:
        print("Erro: Out of memory na GPU ao gerar a resposta. Tente uma imagem menor ou reduza 'max_new_tokens'.")
    except Exception as e:
        print(f"Erro durante a geração da resposta: {e}")
    print("Função 'ask_medgemma_multimodal_new' definida com sucesso.")


ETAPA 4: Definindo a função 'ask_medgemma_multimodal_new'...


In [ ]:
# Testando a capacidade de análise de imagem
print("""\n\n--- ETAPA 5: TESTANDO ANÁLISE DE IMAGEM (NOVO MÉTODO) COM
MEDGEMMA ---""")
url_da_imagem_exemplo = "https://dentistalegal.com.br/wp-content/uploads/2021/06/herpes_labial_dentista_crianca_porto_alegre.jpg" # boca
# url_da_imagem_exemplo = "https://dentistalegal.com.br/wp-content/uploads/2021/06/herpes_labial_dentista_crianca_porto_alegre.jpg" # boca
# Seu prompt de usuário para a análise
prompt_usuario_analise = f"""
Analise esta imagem médica. Descreva as principais características
visíveis.
Se possível, com base nas características observadas, mencione
quaisquer observações ou achados que possam ser relevantes do ponto
de vista médico.
Seja detalhado em sua descrição visual, e se identificar algo,
sugira possíveis diagnósticos e próximos passos ou tratamentos.
"""
# Prompt do sistema (opcional, mas bom para definir o papel da IA)
prompt_sistema = "Você é um assistente de IA especializado em saúde coletiva, análise de imagens médicas."
print(f"\nURL da imagem para análise: {url_da_imagem_exemplo}")
print(f"Prompt do sistema: {prompt_sistema}")
print(f"Prompt do usuário para análise da imagem:\n{prompt_usuario_analise}")
ask_medgemma_multimodal_new(prompt_usuario_analise, url_da_imagem_exemplo, system_prompt=prompt_sistema, max_new_tokens=700)
print("""\n\n--- FIM DO TESTE DE ANÁLISE DE IMAGEM (NOVO MÉTODO) ---""")
print("Lembre-se: As respostas do modelo são geradas por IA e destinam-se a fins de demonstração.")
print("A análise de imagens médicas por IA requer validação rigorosa e não deve substituir o diagnóstico de profissionais de saúde.")



--- ETAPA 5: TESTANDO ANÁLISE DE IMAGEM (NOVO MÉTODO) COM
MEDGEMMA ---

URL da imagem para análise: https://dentistalegal.com.br/wp-content/uploads/2021/06/herpes_labial_dentista_crianca_porto_alegre.jpg
Prompt do sistema: Você é um assistente de IA especializado em saúde coletiva, análise de imagens médicas.
Prompt do usuário para análise da imagem:

Analise esta imagem médica. Descreva as principais características
visíveis.
Se possível, com base nas características observadas, mencione
quaisquer observações ou achados que possam ser relevantes do ponto
de vista médico.
Seja detalhado em sua descrição visual, e se identificar algo,
sugira possíveis diagnósticos e próximos passos ou tratamentos.


 Enviando pergunta multimodal (imagem de URL: https://dentistalegal.com.br/wp-content/uploads/2021/06/herpes_labial_dentista_crianca_porto_alegre.jpg) para MedGemma (aguarde)...
Baixando imagem de: https://dentistalegal.com.br/wp-content/uploads/2021/06/herpes_labial_dentista_crianca_porto